554 Project 3

by Joshua McClure

Fitting Your Model (50 pts)

In [1]:
# Library for Packages
import pandas as pd
import time
import os
from pyspark.ml import Pipeline
from pyspark.sql import SparkSession
from pyspark.sql.types import StructType, StructField, DoubleType, StringType, IntegerType
from pyspark.ml.feature import SQLTransformer, Binarizer, StringIndexer, OneHotEncoder, VectorAssembler, PCA
from pyspark.ml.regression import LinearRegression
from pyspark.ml.tuning import ParamGridBuilder, CrossValidator
from pyspark.ml.evaluation import RegressionEvaluator
from pyspark.sql.functions import col

Part 1: Read in and Organize Data

Create a Jupyter notebook for the modeling fitting part and the Streaming part below.

* The file power_ml_data.csv is available at the URL: https://www4.stat.ncsu.edu/~online/datasets/ power_ml_data.csv

* You should read this data into a standard pandas data frame using the pd.read_csv() function.

* Convert this to a spark data frame

* We are going to treat the Power_Zone_3 variable as our response variable.

* We can use all of the other variables as predictors. (Imagine we know that the Power_Zone_3 reading is going to go offline in the future and we need to be able to predict that value appropriately.)

In [2]:
# Initialize SparkSession
spark = SparkSession.builder.appName("PowerDataProcessing").getOrCreate()

# Define the URL for the dataset
data_url = "https://www4.stat.ncsu.edu/~online/datasets/power_ml_data.csv"

# Read the data into a pandas DataFrame, using the first row as headers
pd_df = pd.read_csv(data_url, header=0)

# Define the Spark schema based on the provided headers
spark_schema = StructType([
    StructField("Temperature", DoubleType(), True),
    StructField("Humidity", DoubleType(), True),
    StructField("Wind_Speed", DoubleType(), True),
    StructField("General_Diffuse_Flows", DoubleType(), True),
    StructField("Diffuse_Flows", DoubleType(), True),
    StructField("Power_Zone_1", DoubleType(), True),
    StructField("Power_Zone_2", DoubleType(), True),
    StructField("Power_Zone_3", DoubleType(), True),
    StructField("Month", IntegerType(), True),
    StructField("Hour", IntegerType(), True)
])

# Convert pandas DataFrame to Spark DataFrame
spark_df = spark.createDataFrame(pd_df, schema=spark_schema)

# Rename Power_Zone_3 to 'label' as it is our response variable
spark_df = spark_df.withColumnRenamed("Power_Zone_3", "label")

# Display schema and first few rows to verify
print("Spark DataFrame Schema:")
spark_df.printSchema()

print("\nFirst 5 rows of Spark DataFrame:")
spark_df.show(5)

Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
26/04/30 12:19:37 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
26/04/30 12:19:38 WARN Utils: Service 'SparkUI' could not bind on port 4040. Attempting port 4041.
26/04/30 12:19:38 WARN Utils: Service 'SparkUI' could not bind on port 4041. Attempting port 4042.
26/04/30 12:19:38 WARN Utils: Service 'SparkUI' could not bind on port 4042. Attempting port 4043.


Spark DataFrame Schema:
root
 |-- Temperature: double (nullable = true)
 |-- Humidity: double (nullable = true)
 |-- Wind_Speed: double (nullable = true)
 |-- General_Diffuse_Flows: double (nullable = true)
 |-- Diffuse_Flows: double (nullable = true)
 |-- Power_Zone_1: double (nullable = true)
 |-- Power_Zone_2: double (nullable = true)
 |-- label: double (nullable = true)
 |-- Month: integer (nullable = true)
 |-- Hour: integer (nullable = true)


First 5 rows of Spark DataFrame:


+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|Temperature|Humidity|Wind_Speed|General_Diffuse_Flows|Diffuse_Flows|Power_Zone_1|Power_Zone_2|      label|Month|Hour|
+-----------+--------+----------+---------------------+-------------+------------+------------+-----------+-----+----+
|      6.559|    73.8|     0.083|                0.051|        0.119|  34055.6962| 16128.87538|20240.96386|    1|   0|
|      6.414|    74.5|     0.083|                 0.07|        0.085| 29814.68354| 19375.07599|20131.08434|    1|   0|
|      6.313|    74.5|      0.08|                0.062|          0.1| 29128.10127| 19006.68693|19668.43373|    1|   0|
|      6.121|    75.0|     0.083|                0.091|        0.096| 28228.86076| 18361.09422|18899.27711|    1|   0|
|      5.921|    75.7|     0.081|                0.048|        0.085|  27335.6962| 17872.34043|18442.40964|    1|   0|
+-----------+--------+----------+---------------

Part 2: Elastic Model Initial Pipeline Set Up

We want to fit an elastic net model using CV (no training/test split, just using CV on the data we’ve read
in) with the steps below.

The transformations below should each use an MLlib function that can be put into a pipeline

  * The Hour column is likely not stored as a DoubleType. If it is not, use an SQL transformer to cast the variable as a DoubleType

  * Binarize the Hour column based on the column being less than 6.5 or not (night vs day essentially)

  * One-hot encode the Month column

  * Run a PCA fit on the Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows columns.

    * To do this, I first used a VectorAssembler() call to place these variables in a column together for use with the PCA() estimator.

    * Once fitted, then you’ll have a PCA transformer we’ll use in our pipeline.

    * We’ll use two PCs in our transformation.

* Rename your response variable as label

* Use VectorAssembler() to put your predictors into a features. Use the:

  * two fitted PCA features

  * binary Hour variable

  * Power_Zone_1

  * Power_Zone_2

  * Month indicator variables

* This ends the pipeline of transformations!

In [3]:
# Step 1: Cast Hour column to DoubleType if not already
sql_transformer_hour = SQLTransformer(statement="SELECT *, CAST(Hour AS DOUBLE) AS Hour_Double FROM __THIS__")

# Step 2: Binarize the Hour column (Night vs Day)
binarizer = Binarizer(threshold=6.5, inputCol="Hour_Double", outputCol="Hour_Binary")

# Step 3: One-hot encode the Month column
month_indexer = StringIndexer(inputCol="Month", outputCol="Month_Indexed")

# Then, OneHotEncoder to convert the indexed column to one-hot vectors
month_encoder = OneHotEncoder(inputCols=["Month_Indexed"], outputCols=["Month_OneHot"])

# Step 4: PCA on Temperature, Humidity, Wind_Speed, General_Diffuse_Flows, and Diffuse_Flows
pca_input_cols = ["Temperature", "Humidity", "Wind_Speed", "General_Diffuse_Flows", "Diffuse_Flows"]
vector_assembler_pca = VectorAssembler(inputCols=pca_input_cols, outputCol="pca_features")

# Apply PCA to reduce dimensions to 2 principal components
pca = PCA(k=2, inputCol="pca_features", outputCol="principal_components")

# Step 5: Final VectorAssembler to put all predictors into a 'features' vector
# Use the two fitted PCA features, binary Hour, Power_Zone_1, Power_Zone_2, and Month indicator variables
final_features_assembler = VectorAssembler(
    inputCols=[
        "principal_components",
        "Hour_Binary",
        "Power_Zone_1",
        "Power_Zone_2",
        "Month_OneHot"
    ],
    outputCol="features"
)

# Step 6: Define the Linear Regression model
lr = LinearRegression(featuresCol="features", labelCol="label", predictionCol="prediction")

# Create the pipeline
pipeline = Pipeline(stages=[
    sql_transformer_hour,
    binarizer,
    month_indexer,
    month_encoder,
    vector_assembler_pca,
    pca,
    final_features_assembler,
    lr # Add the Linear Regression model as the final stage of the pipeline
])

print("Pipeline stages defined successfully.")

Pipeline stages defined successfully.
